# Beginner tour 4b: The classic way, fingerprints and MolPAL's own models

Notebooks 2–4 all used **embeddings** from big pretrained AI models (the backbones). But the
project didn't start there. It started by reproducing a 2021 paper, **MolPAL** (Graff,
Shakhnovich & Coley, *Chemical Science*), whose models use **no pretrained AI at all**:

* a **random forest** and a small **neural network**, both reading hand-designed **fingerprints**
* an **MPN** (message-passing network), a graph neural network trained **from scratch** on the
  molecules' structure

In the project's README this is **method [1]**, the baseline every new idea is compared with.
This notebook answers the obvious beginner question: **why use big AI embeddings at all, if
simpler descriptions exist?** It also runs the part of `run_experiment.py` the other notebooks
skip: `--mode molpal` and the `MolPALExplorer` class.

One cell runs all the full campaigns and takes about 25 minutes (the MPN alone is 15). The rest takes about 1 minute.
**It needs a GPU** for the MPN part (this cluster node has one).

In [ ]:
import sys, json, io, contextlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")            # silence RDKit's many "deprecation" messages
from tour import *

import run_experiment as rx
register_mini(rx)
scores, embeddings = load_mini()
y = -scores["score"].to_numpy(np.float32)       # flipped: higher = better
print("ready")

---
## 1. Fingerprints: a molecule as a checklist

🧪 **Science corner.** A **fingerprint** is a long row of 0s and 1s, a checklist of small
structural features. MolPAL uses **atom-pair** fingerprints: for every pair of atoms, write
down "atom type A, atom type B, this many bonds apart", and switch on one of 2,048 slots for
each such pattern (patterns are assigned to slots by a fixed scrambling rule called
*hashing*). No AI involved: the same molecule always gets the same fingerprint, computed in a
fraction of a millisecond.

The project computes them with `Featurizer` from `molpal/featurizer.py`, the exact settings
the paper used (`pair`, radius 2, 2,048 bits).

🐍 **Python corner: calling an object.** `Featurizer` is a class, but it defines `__call__`, so
an object made from it can be called like a function: `featurize(smiles)`.

In [ ]:
from molpal.featurizer import Featurizer, feature_matrix

featurize = Featurizer(fingerprint="pair", radius=2, length=2048)
aspirin = "CC(=O)Oc1ccccc1C(=O)O"
fp = featurize(aspirin)
print("length:", len(fp), "  slots switched on:", int(fp.sum()), "  which ones:", np.flatnonzero(fp)[:12].tolist(), "...")

Now all 21,044 molecules. `feature_matrix` is the project's helper that featurizes a list:

In [ ]:
t0 = time.time()
F = np.array(feature_matrix(list(scores["smiles"]), featurize))
print(F.shape, f"in {time.time() - t0:.1f} s;  on average {F.sum(1).mean():.0f} of 2,048 slots are on")

How similar are two fingerprints? The standard measure is **Tanimoto similarity**: of all the
slots on in *either* molecule, what fraction are on in *both*? 1 = identical checklists,
0 = nothing in common. Let's find the best molecule's nearest neighbours this way, as we
did with embeddings in notebook 1:

In [ ]:
on = F.astype(bool)
best = int(np.argmax(y))
tanimoto = (on & on[best]).sum(1) / (on | on[best]).sum(1)
closest = np.argsort(-tanimoto)[:5]
draw(scores["smiles"][closest], labels=[f"{scores['score'][i]}  (Tanimoto {tanimoto[i]:.2f})" for i in closest], per_row=5)

In [ ]:
neighbours = np.argsort(-tanimoto)[1:11]
print(f"10 nearest neighbours of the best molecule: average score {scores['score'][neighbours].mean():.2f}"
      f"   (all molecules: {scores['score'].mean():.2f})")

Fingerprint neighbours work too. The 10 molecules whose checklists most resemble the best molecule's average −9.3, far better than the pool's −7.9, and the pictures show why: they share the same building blocks. Notice the Tanimoto values are spread out (about 0.65–0.7 for the closest), unlike Uni-Mol2's similarities, which are all near 1 (see the answers to notebook 1). For fingerprints the actual number means something: 0.7 = 70% of the features in common.

---
## 2. How good are fingerprint-based guesses?

The same fair test as notebook 3: train on 420 random molecules, then measure on the other
20,624 how well the guesses put molecules in the right **order** (Spearman correlation).

We use the project's real random-forest model, `RFModel` from `molpal/models/sklmodels.py`.
🧪 A **random forest** is 100 small decision trees ("if slot 517 is on and slot 1,203 is off,
then...") whose answers are averaged. Their **disagreement** gives an uncertainty, like the
ensembles in notebook 4.

🐍 **Python corner: the same interface again.** Like every model in `molpal`, `RFModel` has
`train(...)` and `get_means_and_vars(...)`. Its `train` takes SMILES plus a `featurizer`, and
does the fingerprinting itself.

In [ ]:
from molpal.models.sklmodels import RFModel

def spearman(a, b):
    return pd.Series(a).corr(pd.Series(b), method="spearman")

order = np.random.default_rng(0).permutation(len(y))
train, test = order[:420], order[420:]

rows = []
rf = RFModel(model_seed=0)
rf.train(list(scores["smiles"][train]), y[train], featurizer=featurize)    # SMILES in, it featurizes
mu, var = rf.get_means_and_vars(list(F[test]))                             # predicts from fingerprints
rows.append({"input": "fingerprints", "model": "random forest (MolPAL)", "test Spearman": spearman(mu, y[test])})

for name in ["unimol2", "mhgged", "molformer"]:                            # the same forest on embeddings
    rf_e = RFModel(model_seed=0)
    rf_e.model.fit(embeddings[name][train], y[train])
    rows.append({"input": f"{name} embedding", "model": "random forest", "test Spearman": spearman(rf_e.model.predict(embeddings[name][test]), y[test])})

table = pd.DataFrame(rows)
table["test Spearman"] = table["test Spearman"].round(3)
table

This puts fingerprints in context:

* With the **same** random forest, fingerprints (0.749) do clearly worse than the Uni-Mol2 and MHG-GED embeddings (0.80), but far better than the MoLFormer embedding (0.43). A big pretrained AI model is **not automatically** better than a simple, hand-designed description. MoLFormer loses to a checklist.
* For comparison, notebook 3's neural-network surrogate reached 0.876 on Uni-Mol2. The random forest is a weaker learner on embeddings, so how you learn matters too, not only what you learn from.

---
## 3. The MPN: learning from the molecule's graph

The third MolPAL model doesn't use a fingerprint either. A **message-passing network** (MPN,
from the *chemprop* library) reads the molecule as a **graph**: atoms are dots and bonds are
lines. Each atom starts with a description of itself, then repeatedly "passes messages" to
its neighbours, so after a few steps each atom knows about its surroundings. Finally all the
atoms' descriptions are combined into one guess.

Here is what the MPN actually receives for aspirin, built by the project's chemprop code:

In [ ]:
from molpal.models.chemprop.features import MolGraph
g = MolGraph(aspirin)
print("atoms:", g.n_atoms)
print("bonds, counted in both directions:", g.n_bonds)
print("numbers describing each atom:", len(g.f_atoms[0]), "   each bond:", len(g.f_bonds[0]))

The big difference from everything else in this tour: the MPN **learns its own description**
of molecules, from scratch, using only the molecules docked so far. The backbones learned
theirs once, from millions of molecules, before the campaign started. Learning from scratch
can fit the task better, but it needs more data and much more computing every round.

---
## 4. Full campaigns: classic vs embeddings

Now the real test: complete 5-round campaigns with the project's `main()`, with the same budget
as always (1,260 molecules docked, best 100 to find). We run 3 seeds each, except the MPN: one
MPN campaign takes about **15 minutes** on this GPU, so it gets one seed. **This cell takes
about 25 minutes**, so start it and go for a coffee.

* `--mode molpal --model rf`: random forest on fingerprints
* `--mode molpal --model nn`: small neural network on fingerprints
* `--mode molpal --model mpn --conf-method mve`: the MPN, with a mean-and-variance output (MolPAL's setting)
* for comparison, `--mode mve`: Uni-Mol2 embeddings alone, and LT-All on three backbones (notebook 4)

We also time each campaign, because **cost** is half of this story.

The MPN gets `--ncpu 0`. That makes it load data inside the main program instead of
starting helper processes for every training pass. On a pool this small the helpers' start-up
time would cost far more than the work itself. (`--ncpu 0` only works on a GPU machine; the
project's other speed option, `--mpn-persistent-workers`, is only used by its parallel
version.)

In [ ]:
def run(mode, *options, seed=42):
    out = new_run_dir()
    sys.argv = ["run_experiment.py", "--dataset", "MINI", "--mode", mode, "--acq", "greedy", *options,   # options come last, so they win
                "--init-size", "210", "--batch-size", "210", "--n-rounds", "5", "--topk", "100",
                "--seed", str(seed), "--run-dir", str(out)]
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        rx.main()
    final = json.loads((out / "history.json").read_text())[-1]["topk_recall"]
    return final, time.time() - t0

strategies = {
    "random forest, fingerprints": ("molpal", "--model", "rf"),
    "neural net, fingerprints":    ("molpal", "--model", "nn"),
    "MPN, molecule graph":         ("molpal", "--model", "mpn", "--conf-method", "mve", "--ncpu", "0"),
    "Uni-Mol2 embedding":          ("mve", "--surrogate", "single", "--backbone", "unimol2"),
    "LT-All, 3 embeddings":        ("mve", "--surrogate", "ltall", "--backbones", "molformer", "unimol2", "mhgged"),
}
seeds = {name: [42, 43, 44] for name in strategies}
seeds["MPN, molecule graph"] = [42]          # one campaign takes ~15 minutes, so just one seed

results = {}
for name, args in strategies.items():
    runs = [run(*args, seed=s) for s in seeds[name]]
    results[name] = {"mean recall": np.mean([r for r, _ in runs]), "spread": np.std([r for r, _ in runs]),
                     "seeds": len(runs), "seconds per campaign": np.mean([sec for _, sec in runs])}
    print(f"{name:28s} recall {results[name]['mean recall']:.3f}   {results[name]['seconds per campaign']:5.0f} s per campaign")

In [ ]:
summary = pd.DataFrame(results).T
display(summary.round({"mean recall": 3, "spread": 3, "seconds per campaign": 0}))

fig, ax = plt.subplots(figsize=(6, 4))
for name, row in summary.iterrows():
    ax.scatter(row["seconds per campaign"], row["mean recall"], s=60)
    ax.annotate(name, (row["seconds per campaign"], row["mean recall"]), textcoords="offset points", xytext=(6, -4), fontsize=8)
ax.axhline(1260 / len(y), color="gray", linestyle="--", label="random docking")
ax.set_xscale("log"); ax.set_xlabel("seconds per campaign (log scale)"); ax.set_ylabel("fraction of the best 100 found")
ax.set_ylim(0, 1); ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

🐍 **Python corner: `iterrows()`** walks through a table one row at a time, giving each row's
name and values.

What this shows, on real data:

* **Quality.** The two good-embedding strategies find the most (0.94–0.96). The classic MolPAL models find clearly less with the same budget: the neural net and the MPN about 0.79, the random forest 0.68. All are far above random docking (6%).
* **Cost.** The MPN took about **12 minutes** per campaign, about **90× slower** than the Uni-Mol2 surrogate (8 seconds), because it retrains a graph network from scratch every round. On the full 2.1M pool the project measured the same pattern: its frozen-embedding fusion ran in under 11 minutes, two to three orders of magnitude faster than MPN (`al-molecular/reports/figure4_and_fusion_comparison.html`).
* **A fairness caveat.** The embeddings' cost was paid *before* the campaign: computing them for the full pool takes GPU-days (notebook 7). That cost is paid once and reused by every experiment afterwards, which is the bet the project makes.
* **Why the MPN does worse here than in the paper.** On the full pool, with tens of thousands of docked molecules, MPN reached about 95–98% recall at the largest batch size. Here it learns from at most 1,260 molecules. A network trained from scratch needs a lot of data, while pretrained embeddings bring knowledge from millions of molecules. So the most likely reason for the gap is the **small amount of training data**, the setting where pretraining helps most. That's a reasonable explanation, not one this notebook proves.
* **One seed for the MPN**, so treat its number as rough. Training on a GPU isn't exactly repeatable: the same command gave 0.83 in a test run while this notebook was being written.

---
## 5. How the code differs

`--mode molpal` builds a different explorer class, `MolPALExplorer`. It isn't a subclass of
`MVEExplorer`: it's a separate class with the same loop shape. The difference shows in its
`__init__`:

In [ ]:
import inspect
src = inspect.getsource(rx.MolPALExplorer.__init__).splitlines()
print("\n".join(l for l in src if "featurizer" in l.lower() or "build_model" in l or "def __init__" in l))

Two things to notice:

* There's **no `emb_dict`** (no pre-computed embeddings). Instead there's a `Featurizer`, and the
  model works from SMILES.
* The model comes from molpal's **factory** `model()` in `molpal/models/__init__.py`, which turns
  `--model mpn --conf-method mve` into an `MPNTwoOutputModel`, `--model rf` into an `RFModel`,
  and `--model nn` into an `NNModelTorch`. The same factory idea as `mve()` in notebook 5.

Each round, the pool is predicted by `_chunked_predict_molpal`. For fingerprint models it
fingerprints each 50,000-molecule chunk and then predicts. Models that read SMILES themselves
(the MPN) skip the featurizer:

In [ ]:
print("models that read SMILES themselves:", rx.SELF_FEATURIZING_TYPES)
src = inspect.getsource(rx._chunked_predict_molpal)
print(src[src.index("    if model.type_ in SELF_FEATURIZING_TYPES"):])

---
## Summary

| Approach | Describes a molecule by | Learns | Quality here | Cost per campaign |
|---|---|---|---|---|
| random forest / neural net (MolPAL) | an atom-pair **fingerprint** (fixed checklist) | from the docked molecules only | 0.68 / 0.79 | under a minute |
| MPN (MolPAL) | the molecule's **graph** | its own description, from scratch | ~0.8 (1 seed) | ~12 minutes |
| embedding surrogates (this project) | a **pretrained backbone's** numbers | a small model on top | 0.94–0.96 | seconds, plus a one-time embedding cost |

The project's bet, visible here in miniature: describe molecules **once** with big pretrained
models, then make every active-learning round cheap. It wins when the backbone is good
(Uni-Mol2, MHG-GED). It can lose to a simple fingerprint when the backbone is weak (MoLFormer).

🐍 **Python you met:** calling an object (`__call__`), boolean arrays for set-like maths (`&`,
`|`), the shared `train`/`get_means_and_vars` interface, timing with `time.time()`,
`iterrows`, scatter plots with labels.

---
## Try it yourself
Answers, run on the same real data: [solutions/4b_solutions.ipynb](solutions/4b_solutions.ipynb). Try first!

1. Use a **Morgan** fingerprint instead (`Featurizer(fingerprint="morgan", radius=2, length=2048)`) in section 2. Better or worse than atom-pair?
2. In section 4, run the random forest with `"--acq", "ucb"`. A random forest's uncertainty is how much its 100 trees disagree: does exploring it help?
3. How many campaigns of the **Uni-Mol2** strategy could you run in the time of one **MPN** campaign?

**Next:** `5` is `../00_how_al_molecular_works/classes_and_call_flow.ipynb`.